In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Ekstraksi data dari file .mseed ke format JSON siap untuk MCU-Quake.
- Menggunakan STA/LTA untuk deteksi P-wave arrival.
- Ekstrak 7 detik sinyal dan noise.
- Preprocessing: detrend, resample ke 100 Hz, normalisasi.
- Output: JSON dengan key 'Z' dan 'Z_noise'.
"""

import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from tqdm import tqdm
import logging

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"
OUTPUT_JSON = "/Volumes/Extreme SSD/unduhan_waveform_geofon/extracted_data.json"

# Parameter preprocessing
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0

# Parameter STA/LTA
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5

# Jumlah file untuk testing (None untuk semua)
MAX_FILES = None  # misal 100 untuk testing

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("extract_waveforms.log")
    ]
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI PREPROCESSING
# =============================================

def pick_p_arrival(trace, search_window=15):
    """
    Deteksi P-wave arrival menggunakan STA/LTA.
    Kembalikan UTCDateTime dari pick pertama.
    """
    try:
        # STA/LTA hanya pada trace yang dipotong dari awal
        tr = trace.copy()
        sr = tr.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        
        # Jika trace terlalu pendek, skip
        if len(tr.data) < lta_n + sta_n:
            return tr.stats.starttime + 5.0  # fallback
        
        cft = recursive_sta_lta(tr.data, sta_n, lta_n)
        trigger_indices = np.where(cft > TRIGGER_THRESHOLD)[0]
        
        if len(trigger_indices) > 0:
            pick_idx = trigger_indices[0]
            if pick_idx > int(2 * sr):  # hindari trigger di awal
                return tr.stats.starttime + pick_idx / sr
        
        # Fallback: ambil puncak maksimum
        max_idx = np.argmax(np.abs(tr.data))
        if max_idx > 0:
            return tr.stats.starttime + max_idx / sr
    except Exception as e:
        logger.debug(f"Picking error: {e}")
        return trace.stats.starttime + 5.0
    
    return trace.stats.starttime + 5.0

def extract_mcuquake_windows(trace, p_arrival_time):
    """
    Ekstraksi 7 detik sinyal dan noise, preprocessing.
    Return (signal_list, noise_list) atau (None, None) jika gagal.
    """
    try:
        # Potong sinyal 7 detik setelah P
        sig_start = p_arrival_time
        sig_end = p_arrival_time + SIG_DURATION
        tr_signal = trace.copy().trim(sig_start, sig_end)
        
        # Potong noise 7 detik sebelum P
        noise_start = p_arrival_time - NOISE_DURATION
        noise_end = p_arrival_time
        tr_noise = trace.copy().trim(noise_start, noise_end)
        
        # Detrend
        tr_signal.detrend('simple')
        tr_noise.detrend('simple')
        
        # Resample ke 100 Hz
        if tr_signal.stats.sampling_rate != SAMPLE_RATE:
            tr_signal.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)
        
        # Normalisasi dengan max absolut 9 detik setelah P
        tr_norm = trace.copy().trim(p_arrival_time, p_arrival_time + NORM_WINDOW)
        if len(tr_norm.data) > 0:
            max_val = np.max(np.abs(tr_norm.data))
        else:
            max_val = np.max(np.abs(tr_signal.data))
        if max_val == 0:
            max_val = 1.0
        
        signal_data = tr_signal.data / max_val
        noise_data = tr_noise.data / max_val
        
        # Potong/padding ke 700 sampel
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data
        
        return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()
    except Exception as e:
        logger.debug(f"Extraction error: {e}")
        return None, None

def process_file(file_path):
    """
    Proses satu file .mseed, return dict hasil atau None.
    """
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None
        
        # Ambil komponen Z (prioritas BHZ, HHZ, EHZ)
        trace_z = None
        for tr in st:
            if tr.stats.channel.endswith('Z'):
                trace_z = tr
                break
        if trace_z is None:
            # Jika tidak ada Z, ambil trace pertama
            trace_z = st[0]
            logger.debug(f"{file_path.name}: Tidak ada komponen Z, pakai {trace_z.stats.channel}")
        
        # Deteksi P-wave
        p_time = pick_p_arrival(trace_z)
        
        # Ekstraksi
        signal, noise = extract_mcuquake_windows(trace_z, p_time)
        if signal is None or noise is None:
            return None
        
        # Ambil info dari nama file: GE_TNTI_20100101_044258.mseed
        parts = file_path.stem.split('_')
        if len(parts) >= 3:
            network = parts[0]
            station = parts[1]
            event_id = '_'.join(parts[2:])
        else:
            event_id = file_path.stem
        
        return {
            'event_id': event_id,
            'network': network if 'network' in locals() else 'UNK',
            'station': station if 'station' in locals() else 'UNK',
            'Z': signal,
            'Z_noise': noise,
            'p_arrival': str(p_time),
            'file': file_path.name
        }
    except Exception as e:
        logger.debug(f"Error processing {file_path.name}: {e}")
        return None

def main():
    logger.info("="*60)
    logger.info("🚀 EKSTRAKSI WAVEFORM KE JSON (MCU-QUAKE)")
    logger.info("="*60)
    
    # Cari semua file .mseed
    wave_dir = Path(WAVEFORM_DIR)
    all_files = list(wave_dir.glob("*.mseed"))
    logger.info(f"📁 Ditemukan {len(all_files)} file .mseed")
    
    if MAX_FILES and len(all_files) > MAX_FILES:
        all_files = all_files[:MAX_FILES]
        logger.info(f"⚠️ Hanya memproses {MAX_FILES} file pertama.")
    
    # Load JSON yang sudah ada (resume)
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r') as f:
            existing_data = json.load(f)
        logger.info(f"📂 Load JSON existing: {len(existing_data)} entries")
    else:
        existing_data = {}
    
    # Proses file
    success = 0
    failed = 0
    skipped = 0
    
    for file_path in tqdm(all_files, desc="Memproses"):
        # Cek apakah file sudah ada di JSON
        # Gunakan nama file sebagai key (atau bagian dari event_id)
        if file_path.stem in existing_data:
            skipped += 1
            continue
        
        result = process_file(file_path)
        if result:
            # Gunakan event_id + stasiun sebagai key agar unik
            key = f"{result['event_id']}_{result['station']}"
            existing_data[key] = {
                'type': 'se',
                'Z': result['Z'],
                'Z_noise': result['Z_noise'],
                'metadata': {
                    'network': result['network'],
                    'station': result['station'],
                    'p_arrival': result['p_arrival'],
                    'file': result['file']
                }
            }
            success += 1
        else:
            failed += 1
        
        # Simpan setiap 100 file untuk menghindari kehilangan data
        if (success + failed) % 100 == 0:
            with open(OUTPUT_JSON, 'w') as f:
                json.dump(existing_data, f, indent=2)
    
    # Simpan final
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(existing_data, f, indent=2)
    
    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Skipped: {skipped}")
    logger.info(f"📁 Total data di JSON: {len(existing_data)}")
    logger.info(f"📂 Output: {OUTPUT_JSON}")
    logger.info("="*60)

if __name__ == "__main__":
    main()